# Neural Functional Theory for 1D Hard Rods — Python/PyTorch Tutorial

This notebook is a Python/PyTorch port of the Julia [NeuralDFT-Tutorial](https://github.com/sfalmo/NeuralDFT-Tutorial) by Sammüller, Hermann, de las Heras & Schmidt  
([J. Phys.: Condens. Matter **36**, 243002, 2024](https://doi.org/10.1088/1361-648X/ad326f)).

We cover the full pipeline:
1. **Grand-canonical Monte Carlo (GCMC)** — generate exact density profiles ρ(x) for 1D hard rods
2. **Percus' exact functional** — the analytical DFT solution for 1D hard rods
3. **Neural functional theory** — train an MLP to learn ρ_window → c₁(x), then use it for DFT minimisation and compute c₂ via autograd

**System:** 1D hard rods, diameter σ = 1, periodic box of length L.  
Hard-core interaction: ϕ(r) = ∞ if |r| < σ, 0 otherwise.  
Grand-canonical ensemble: temperature T = 1, chemical potential μ (in β-units), β = 1/T = 1.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Reproducible RNG
RNG = np.random.default_rng(42)

---
## Part 1: Grand-Canonical Monte Carlo

### The model

Hard rods in 1D: $N$ point-like particles of diameter σ = 1 in a periodic box of length $L$.  
Particles at positions $\{x_i\}$ interact via:
$$\phi(|x_i - x_j|) = \begin{cases} \infty & |x_i - x_j| < \sigma \\ 0 & \text{otherwise} \end{cases}$$

An external potential $V_\mathrm{ext}(x)$ can bias the density profile.

### Grand-canonical MC moves

| Move | Acceptance |
|------|------------|
| Insert at random $x$ | $\min(1,\; L/N_{\rm new} \cdot e^{\beta\mu_{\rm loc}(x)})$ |
| Delete particle $i$   | $\min(1,\; N_{\rm old}/L \cdot e^{-\beta\mu_{\rm loc}(x_i)})$ |
| Translate $\pm\Delta x$ | 1 if no overlap, else 0 |

where $\beta\mu_{\rm loc}(x) = \mu - V_\mathrm{ext}(x)$ (all in β-units).

In [ ]:
from mlip_mc.dft.hard_rod_sim import HardRodSystem, simulate, generate_random_vext

# System parameters
L = 10.0    # box length in σ units
mu = 1.5    # chemical potential β·μ
T = 1.0     # temperature (kB = 1)
n_bins = 500

# Sinusoidal external potential
def vext_sin(x):
    """Sinusoidal external potential (in β-units)."""
    return 1.5 * np.sin(2 * np.pi * x / L)

print("Running GCMC simulation...")
x_sim, rho_sim, mu_loc_sim = simulate(
    L=L, mu=mu, T=T, vext_fn=vext_sin,
    n_bins=n_bins, n_equil=20_000, n_prod=200_000,
    rng=RNG
)
print(f"  Mean density: {rho_sim.mean():.4f} (avg. N = {rho_sim.mean() * L:.1f})")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

axes[0].plot(x_sim, rho_sim, 'b-', lw=1.5, label=r'$\rho(x)$ GCMC')
axes[0].set_ylabel(r'Density $\rho(x)$  [σ$^{-1}$]')
axes[0].legend()
axes[0].set_title(f'1D Hard-Rod GCMC  (L={L}, μ={mu}, T={T})')

vext_vals = np.array([vext_sin(x) for x in x_sim])
axes[1].plot(x_sim, vext_vals, 'r-', lw=1.5, label=r'$V_{\rm ext}(x)/k_BT$')
axes[1].plot(x_sim, mu_loc_sim, 'g--', lw=1.5, label=r'$\mu_{\rm loc}(x) = \mu - V_{\rm ext}$')
axes[1].set_xlabel(r'Position $x$ [σ]')
axes[1].set_ylabel(r'Potential [$k_BT$]')
axes[1].legend()

plt.tight_layout()
plt.show()

### One-body direct correlation from simulation

The Euler–Lagrange equation of DFT (at self-consistency) gives:
$$\ln \rho(x) = \beta\mu_{\rm loc}(x) + c_1(x; [\rho])$$

Rearranging, we can extract $c_1$ directly from the simulation:
$$c_1(x) = \ln \rho(x) - \beta\mu_{\rm loc}(x)$$

This is the "exact" $c_1$ consistent with the sampled density.

In [ ]:
# Compute c1 from simulation
# Only meaningful where rho > 0
mask = rho_sim > 1e-8
c1_sim = np.where(mask, np.log(np.maximum(rho_sim, 1e-12)) - mu_loc_sim, np.nan)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(x_sim[mask], c1_sim[mask], 'b.', ms=2, alpha=0.7, label=r'$c_1$ from GCMC')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel(r'Position $x$ [σ]')
ax.set_ylabel(r'$c_1(x)$')
ax.set_title(r'One-body direct correlation $c_1(x) = \ln\rho(x) - \beta\mu_{\rm loc}(x)$')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 2: Percus' Exact Functional

For 1D hard rods, Percus (1976) derived the **exact** free-energy functional.  
The one-body direct correlation is:
$$c_1(x) = -\ln(1 - n_1(x))$$

where $n_1(x)$ is the weighted density:
$$n_1(x) = \int_{x-R}^{x+R} \rho(x')\, dx', \quad R = \sigma/2$$

This is evaluated via FFT convolution with the step-function weight $\omega_1(x) = \Theta(R - |x|)$.

### DFT self-consistency (Picard iteration)

Given $c_1$, the equilibrium density satisfies:
$$\rho(x) = \exp\bigl(\beta\mu_{\rm loc}(x) + c_1(x;[\rho])\bigr)$$

We iterate (Picard mixing):
$$\rho \leftarrow (1-\alpha)\,\rho + \alpha\,\rho_{\rm new}$$
until convergence.

In [ ]:
from mlip_mc.dft.percus import c1_percus, dft_minimize

# Test Percus c1 on the simulated density profile
c1_percus_from_sim = c1_percus(rho_sim, x_sim, sigma=1.0)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(x_sim[mask], c1_sim[mask], 'b.', ms=2, alpha=0.6, label=r'$c_1$ from GCMC')
ax.plot(x_sim, c1_percus_from_sim, 'r-', lw=1.5, label=r'Percus $c_1[\rho_{\rm GCMC}]$')
ax.set_xlabel(r'Position $x$ [σ]')
ax.set_ylabel(r'$c_1(x)$')
ax.set_title('Percus functional applied to GCMC density')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("Running Percus DFT minimisation (Picard iteration)...")

c1_fn_percus = lambda rho, xs: c1_percus(rho, xs, sigma=1.0)

x_dft, rho_dft = dft_minimize(
    L=L, mu=mu, T=T, vext_fn=vext_sin,
    c1_fn=c1_fn_percus,
    n_bins=n_bins, alpha=0.05, max_iter=5000, tol=1e-7
)

print(f"  DFT mean density: {rho_dft.mean():.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_sim, rho_sim, 'b-', lw=1.5, alpha=0.8, label='GCMC (exact)')
ax.plot(x_dft, rho_dft, 'r--', lw=2, label='Percus DFT')
ax.set_xlabel(r'Position $x$ [σ]')
ax.set_ylabel(r'Density $\rho(x)$ [σ$^{-1}$]')
ax.set_title('GCMC vs Percus DFT')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Try a random external potential
vext_random = generate_random_vext(L, rng=RNG, n_sin=4, amplitude=1.5)

print("GCMC with random Vext...")
x_r, rho_r, mu_loc_r = simulate(
    L=L, mu=2.0, T=T, vext_fn=vext_random,
    n_bins=n_bins, n_equil=20_000, n_prod=200_000,
    rng=RNG
)

print("Percus DFT with same Vext...")
x_r_dft, rho_r_dft = dft_minimize(
    L=L, mu=2.0, T=T, vext_fn=vext_random,
    c1_fn=c1_fn_percus,
    n_bins=n_bins, alpha=0.05, max_iter=8000
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x_r, rho_r, 'b-', lw=1.5, alpha=0.8, label='GCMC')
axes[0].plot(x_r_dft, rho_r_dft, 'r--', lw=2, label='Percus DFT')
axes[0].set_xlabel(r'$x$ [σ]')
axes[0].set_ylabel(r'$\rho(x)$')
axes[0].set_title('Random Vext: GCMC vs DFT')
axes[0].legend()

vext_vals_r = np.array([vext_random(x) for x in x_r])
axes[1].plot(x_r, vext_vals_r, 'g-', lw=1.5)
axes[1].set_xlabel(r'$x$ [σ]')
axes[1].set_ylabel(r'$\beta V_{\rm ext}(x)$')
axes[1].set_title('External potential')

plt.tight_layout()
plt.show()

---
## Part 3: Neural Functional Theory

### Concept

Instead of using the analytical Percus functional, we train a neural network to approximate $c_1$:

$$c_1(x) \approx f_\theta\!\left(\rho(x-w/2),\, \ldots,\, \rho(x+w/2)\right)$$

where $f_\theta$ is an MLP that takes a **local density window** of width $w$ centred at $x$ as input and outputs a scalar $c_1(x)$.

This "machine-learned functional" can then be used in DFT minimisation exactly like the Percus functional, and differentiated to obtain the two-body direct correlation $c_2(x,x') = \delta c_1(x)/\delta\rho(x')$.

### Training strategy

1. Run many GCMC simulations with random $\mu$ and random $V_\mathrm{ext}$.
2. Extract $(\rho, c_1)$ pairs from each simulation using $c_1 = \ln\rho - \beta\mu_{\rm loc}$.
3. Build a dataset of (window, $c_1$) pairs and train the MLP with Adam.
4. Augment with mirror-flipped profiles for spatial symmetry.

### Step 3.1 — Generate training data

In [ ]:
from mlip_mc.dft.neural_functional import NeuralC1Functional, generate_training_data

print("Generating training data...")
print("(Each sample runs a full GCMC simulation with random Vext and random μ)")
print()

data = generate_training_data(
    n_samples=200,
    L=10.0,
    mu_range=(-0.5, 2.5),
    n_bins=500,
    n_equil=5_000,
    n_prod=50_000,
    rng=np.random.default_rng(0),
    verbose=True,
)
print(f"\nDataset size: {len(data)} profiles")

In [ ]:
# Visualise a few training samples
fig, axes = plt.subplots(3, 2, figsize=(12, 9))
sample_indices = np.random.default_rng(7).choice(len(data), 3, replace=False)

for row, idx in enumerate(sample_indices):
    d = data[idx]
    mask_i = d['rho'] > 1e-8
    axes[row, 0].plot(d['x'], d['rho'], 'b-', lw=1.2)
    axes[row, 0].set_ylabel(r'$\rho(x)$')
    if row == 2:
        axes[row, 0].set_xlabel(r'$x$ [σ]')
    axes[row, 0].set_title(f'Sample {idx}: density')

    axes[row, 1].plot(d['x'][mask_i], d['c1'][mask_i], 'r.', ms=1.5, alpha=0.7)
    axes[row, 1].set_ylabel(r'$c_1(x)$')
    if row == 2:
        axes[row, 1].set_xlabel(r'$x$ [σ]')
    axes[row, 1].set_title(f'Sample {idx}: $c_1$')

plt.suptitle('Training data samples', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Step 3.2 — Train the neural functional

In [ ]:
# Extract lists of rho and c1 profiles
dx_data = data[0]['x'][1] - data[0]['x'][0]

rho_profiles = [d['rho'] for d in data]
c1_profiles  = [d['c1']  for d in data]

# Build the neural functional
# window_width = 1.5σ gives a window of ±0.75σ, slightly larger than Percus range (±0.5σ)
neural_c1 = NeuralC1Functional(
    window_width=1.5,     # [σ]
    dx=dx_data,
    hidden_dims=[64, 64, 64]
)
neural_c1.build_model()

print(f"Window width: {neural_c1.window_width} σ  →  {neural_c1.n_window_bins} bins per window")
print(f"Training on {len(rho_profiles)} profiles...\n")

history = neural_c1.train(
    rho_profiles=rho_profiles,
    c1_profiles=c1_profiles,
    epochs=300,
    lr=1e-3,
    batch_size=1024,
    val_fraction=0.1,
    verbose=True,
)

In [ ]:
# Plot training loss curve
fig, ax = plt.subplots(figsize=(8, 4))
epochs_range = np.arange(1, len(history['train_loss']) + 1)
ax.semilogy(epochs_range, history['train_loss'], 'b-', lw=1.5, label='Train MSE')
ax.semilogy(epochs_range, history['val_loss'],   'r-', lw=1.5, label='Val MSE')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Neural functional training')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final train MSE: {history['train_loss'][-1]:.4e}")
print(f"Final val   MSE: {history['val_loss'][-1]:.4e}")

### Step 3.3 — DFT minimisation with the neural functional

In [ ]:
from mlip_mc.dft.percus import dft_minimize

# Define a new test potential (not in training set)
def vext_test(x):
    return 2.0 * np.cos(2 * np.pi * x / L) + 0.8 * np.sin(4 * np.pi * x / L + 0.5)

mu_test = 1.8

# --- GCMC reference ---
print("GCMC reference simulation...")
x_test, rho_test_gcmc, mu_loc_test = simulate(
    L=L, mu=mu_test, T=T, vext_fn=vext_test,
    n_bins=n_bins, n_equil=20_000, n_prod=300_000,
    rng=np.random.default_rng(99)
)

# --- Percus DFT ---
print("Percus DFT...")
_, rho_test_percus = dft_minimize(
    L=L, mu=mu_test, T=T, vext_fn=vext_test,
    c1_fn=c1_fn_percus,
    n_bins=n_bins, alpha=0.05, max_iter=8000
)

# --- Neural DFT ---
print("Neural DFT...")
def c1_fn_neural(rho, xs):
    return neural_c1.predict(rho)

_, rho_test_neural = dft_minimize(
    L=L, mu=mu_test, T=T, vext_fn=vext_test,
    c1_fn=c1_fn_neural,
    n_bins=n_bins, alpha=0.03, max_iter=10_000, tol=1e-5
)

print("Done.")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axes[0].plot(x_test, rho_test_gcmc,   'b-',  lw=1.5, alpha=0.85, label='GCMC (exact)')
axes[0].plot(x_test, rho_test_percus, 'g--', lw=2,   label='Percus DFT (exact)')
axes[0].plot(x_test, rho_test_neural, 'r:',  lw=2.5, label='Neural DFT')
axes[0].set_ylabel(r'$\rho(x)$ [σ$^{-1}$]')
axes[0].set_title(f'Test system (μ={mu_test}, new Vext)')
axes[0].legend()

# Residuals
axes[1].plot(x_test, rho_test_percus - rho_test_gcmc, 'g--', lw=1.5, label='Percus − GCMC')
axes[1].plot(x_test, rho_test_neural - rho_test_gcmc, 'r:',  lw=1.5, label='Neural − GCMC')
axes[1].axhline(0, color='k', lw=0.8, ls='-')
axes[1].set_xlabel(r'$x$ [σ]')
axes[1].set_ylabel(r'$\Delta\rho(x)$')
axes[1].set_title('Residuals')
axes[1].legend()

plt.tight_layout()
plt.show()

rmse_percus = np.sqrt(np.mean((rho_test_percus - rho_test_gcmc)**2))
rmse_neural = np.sqrt(np.mean((rho_test_neural - rho_test_gcmc)**2))
print(f"RMSE Percus: {rmse_percus:.4f}")
print(f"RMSE Neural: {rmse_neural:.4f}")

### Step 3.4 — Two-body direct correlation c₂ via autograd

The two-body direct correlation function is the functional derivative of $c_1$:
$$c_2(x, x') = \frac{\delta c_1(x)}{\delta \rho(x')}$$

For the neural functional, this is simply the Jacobian of the MLP output with respect to the density, computed via PyTorch autograd.  

For Percus' exact functional:
$$c_2^{\rm Percus}(x, x') = -\frac{\omega_1(x'-x)}{1-n_1(x)}$$

This is a short-ranged function (zero for $|x-x'| > \sigma$).

In [ ]:
# Compute c2 at a few reference positions using the neural functional
# We use the converged neural DFT density as input
rho_for_c2 = rho_test_neural
dx_c2 = L / n_bins
x_for_c2 = np.linspace(dx_c2 / 2, L - dx_c2 / 2, n_bins)

# Pick three reference positions
ref_indices = [n_bins // 4, n_bins // 2, 3 * n_bins // 4]
ref_positions = [x_for_c2[i] for i in ref_indices]

print("Computing c₂(x_ref, x') via autograd for reference positions:")
c2_profiles = []
for i, idx in enumerate(ref_indices):
    print(f"  x_ref = {ref_positions[i]:.2f} σ  (index {idx})")
    c2 = neural_c1.c2_at(rho_for_c2, idx)
    c2_profiles.append(c2)

print("Done.")

In [ ]:
fig, axes = plt.subplots(1, len(ref_indices), figsize=(13, 4), sharey=True)

colors = ['steelblue', 'darkorange', 'seagreen']
for ax, c2, x_ref, col in zip(axes, c2_profiles, ref_positions, colors):
    ax.plot(x_for_c2, c2, color=col, lw=1.5)
    ax.axvline(x_ref, color='k', ls='--', lw=1, label=f'$x_{{\\rm ref}}={x_ref:.1f}$')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xlabel(r"$x'$ [σ]")
    ax.set_title(f"$c_2({x_ref:.1f},\, x')$")
    ax.legend(fontsize=9)

axes[0].set_ylabel(r"$c_2(x_{\rm ref},\, x')$")
plt.suptitle('Two-body direct correlation via autograd (neural functional)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Compute a c2 heatmap for a subset of reference positions
# (subsample for speed: every 10th point)
subsample = 10
n_sub = n_bins // subsample
ref_idxs_heat = np.arange(0, n_bins, subsample)

print(f"Computing c₂ heatmap ({n_sub} × {n_bins})...")
C2_matrix = np.zeros((n_sub, n_bins))
for row_i, idx in enumerate(ref_idxs_heat):
    C2_matrix[row_i] = neural_c1.c2_at(rho_for_c2, int(idx))

print("Done.")

x_sub = x_for_c2[ref_idxs_heat]

fig, ax = plt.subplots(figsize=(8, 6))
vmax = np.percentile(np.abs(C2_matrix), 98)
im = ax.pcolormesh(
    x_for_c2, x_sub, C2_matrix,
    cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto'
)
plt.colorbar(im, ax=ax, label=r"$c_2(x, x')$")
ax.set_xlabel(r"$x'$ [σ]")
ax.set_ylabel(r"$x$ [σ]")
ax.set_title(r"Two-body direct correlation $c_2(x, x')$ — neural functional via autograd")
plt.tight_layout()
plt.show()

### Step 3.5 — Save and reload the model

In [ ]:
import os, tempfile

save_path = os.path.join(tempfile.gettempdir(), 'neural_c1_hard_rod.pt')
neural_c1.save(save_path)
print(f"Model saved to: {save_path}")

# Reload and verify
neural_c1_loaded = NeuralC1Functional.load(save_path)
c1_reloaded = neural_c1_loaded.predict(rho_test_gcmc)
c1_original = neural_c1.predict(rho_test_gcmc)
print(f"Max difference after reload: {np.max(np.abs(c1_reloaded - c1_original)):.2e}  (should be ~0)")

---
## Summary

| Method | Description | Accuracy |
|--------|-------------|----------|
| **GCMC** | Exact (statistical) simulation | Reference |
| **Percus DFT** | Exact functional via FFT convolution | Machine-precision (in principle) |
| **Neural DFT** | MLP $f_\theta$: window → $c_1$ | Controlled by training data size |

Key take-aways:
- The neural functional learns a **universal** approximation to $c_1[\rho]$ from simulation data.
- Once trained, DFT minimisation is cheap (no MC needed for new external potentials).
- Autograd gives the two-body DCF $c_2 = \delta c_1/\delta\rho$ **for free**.
- For 1D hard rods the Percus functional is exact; the neural version is a demonstration of the method for systems where no exact solution exists.

**Next steps:** Apply to 3D systems, use MLIP energies (via `mlip_mc.src`) for the MC sampling, and train neural functionals for realistic fluids.